# Final test: metric comparison chart on half B

**CRISP-DM stage:** Evaluation

The grouped bar chart from the blueprint's list of figures (section 5.7, figure 2), drawn for the reported half B results. It reads `outputs/results/final_test/final_results_table.csv`, which `final_results.ipynb` saved, so no model is run and nothing is chosen.

- **Three panels per product:**
  - AU-PRO at 30 percent, the benchmark's localisation score.
  - Image F1 at the locked cost-based line, with the flag-everything F1 as a dashed line.
  - Image AU-ROC with its 95 percent bootstrap interval, with chance (0.5) as a dashed line.
- **Colours:** the same colour per configuration as in the other final figures.
- **Careful:** half B has only 2 good parts per product, so the intervals are wide.

**Runs on:** the laptop.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt

PROJECT_FOLDER = Path.cwd()
if not (PROJECT_FOLDER / "src").exists():
    PROJECT_FOLDER = PROJECT_FOLDER.parent
TABLE = PROJECT_FOLDER / "outputs/results/final_test/final_results_table.csv"
FIGURE = PROJECT_FOLDER / "outputs/figures/final_metric_comparison.png"
PRODUCTS = ["sheet_metal", "vial"]
RUN_IDS = ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C6_512"]
LABELS = {"C6_512": "C6\n512"}
# The same colour for each configuration as in the other final figures
COLOURS = {"C1": "#2a78d6", "C2": "#eb6834", "C3": "#56b4e9", "C4": "#1baf7a", "C5": "#eda100",
           "C6": "#cc79a7", "C7": "#7a5195", "C8": "#8c6d31", "C6_512": "#cc79a7"}

rows = {}
with open(TABLE, encoding="utf-8") as file:
    for row in csv.DictReader(file):
        rows[(row["product"], row["config"])] = row

figure, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=True)
for row_number in range(len(PRODUCTS)):
    product = PRODUCTS[row_number]
    positions = list(range(len(RUN_IDS)))
    names = []
    colours = []
    for run_id in RUN_IDS:
        names.append(LABELS.get(run_id, run_id))
        colours.append(COLOURS[run_id])

    # Panel 1: AU-PRO at 30 percent
    values = []
    for run_id in RUN_IDS:
        values.append(float(rows[(product, run_id)]["au_pro_30_percent"]))
    axis = axes[row_number][0]
    axis.bar(positions, values, color=colours, width=0.7)
    axis.set_title(product.replace("_", " ").title() + ": AU-PRO at 30%")

    # Panel 2: image F1 at the locked cost-based line, against flagging everything
    values = []
    for run_id in RUN_IDS:
        values.append(float(rows[(product, run_id)]["image_f1_cost_line"]))
    flag_everything = float(rows[(product, "C1")]["flag_everything_f1"])
    axis = axes[row_number][1]
    axis.bar(positions, values, color=colours, width=0.7)
    axis.axhline(flag_everything, color="#333333", linestyle="--", linewidth=1)
    axis.text(len(RUN_IDS) - 0.5, flag_everything + 0.02, "flag everything", ha="right", fontsize=8)
    axis.set_title(product.replace("_", " ").title() + ": image F1, locked cost-based line")

    # Panel 3: image AU-ROC with its bootstrap interval
    values = []
    below = []
    above = []
    for run_id in RUN_IDS:
        row = rows[(product, run_id)]
        value = float(row["image_au_roc"])
        values.append(value)
        below.append(value - float(row["image_au_roc_low"]))
        above.append(float(row["image_au_roc_high"]) - value)
    axis = axes[row_number][2]
    axis.bar(positions, values, color=colours, width=0.7, yerr=[below, above], capsize=3,
             error_kw={"elinewidth": 1, "ecolor": "#333333"})
    axis.axhline(0.5, color="#333333", linestyle="--", linewidth=1)
    axis.text(len(RUN_IDS) - 0.5, 0.52, "chance", ha="right", fontsize=8)
    axis.set_title(product.replace("_", " ").title() + ": image AU-ROC, 95% interval")

    for axis in axes[row_number]:
        axis.set_xticks(positions)
        axis.set_xticklabels(names, fontsize=9)
        axis.set_ylim(0, 1.05)
        axis.grid(axis="y", alpha=0.3)

figure.suptitle("Final test on half B. C1 to C3 PaDiM, C4 to C6 PatchCore, C7 WinCLIP, C8 Reverse Distillation. "
                "Only 2 good parts per product, so intervals are wide")
figure.tight_layout()
figure.savefig(FIGURE, dpi=150)
plt.close(figure)
print("Saved", FIGURE)